# S5 · Evaluación de la Unidad I

**Integrante** · Joel Jonatan Quispe Mamani

Equipo 04 · AeroStreamPeru · Ciclo 9

**Objetivo de negocio.** Pronosticar la velocidad mediana sobre el suelo en la siguiente ventana. El error queda en m/s y ayuda a juzgar si un cambio previsto respecto a la velocidad habitual merece revisión.

## 1. Arquitectura Lambda y alcance del producto

```text
OpenSky histórico → Bronze Parquet → Spark / Silver → Gold por ventana → MLlib y modelo
OpenSky REST → Kafka → Spark en streaming → predicción y tablero Grafana
```

La rama batch de S5 entrega, sobre todo el histórico, el indicador descriptivo y el modelo de la propia dimensión. Silver aplica reglas de esquema, nulos y deduplicación. Gold organiza los indicadores por fecha de la ventana. La rama de velocidad consumirá las mismas definiciones de entrada cuando llegue telemetría nueva.

## 2. Variable objetivo y variables predictoras

El objetivo es **`velocidad_mediana_t1`**, el valor real de `velocidad_mediana_t` en la siguiente ventana de diez minutos. Las predictoras de la lista se conocen al cerrar la ventana actual. La hora y el día de la semana corresponden a Lima. La sección 5 construye la etiqueta y la sección 6 conecta las predictoras con `VectorAssembler`.

Los percentiles 25, 50 y 75 se calculan sobre velocidades válidas en m/s. La altitud barométrica mediana, la velocidad vertical mediana y el volumen promedio describen la misma ventana.

In [1]:
INTEGRANTE = "joel"
INDICADOR_ACTUAL = "velocidad_mediana_t"
VARIABLE_OBJETIVO = "velocidad_mediana_t1"
UNIDAD = "m/s"

VARIABLES_PREDICTORAS = [
    "velocidad_mediana_t",
    "velocidad_p25_t",
    "velocidad_p75_t",
    "altitud_mediana_t",
    "velocidad_vertical_mediana_t",
    "volumen_promedio_t",
    "hora_del_dia",
    "dia_semana",
]

In [2]:
import json
import sys
from pathlib import Path

from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import Imputer, VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import DateType, DoubleType, IntegerType, LongType, StructType, TimestampType

SESION = Path("/opt/s05-evaluacion-unidad-1")
sys.path.insert(0, str(SESION))
from preparar_gold_historico import CADENCIA_SEGUNDOS, limpiar_estados, preparar_historico
from preparar_gold_historico import ESQUEMA as ESQUEMA_BRONZE

HISTORICO = SESION / "artifacts" / "historico"
BRONZE = Path("/opt/s01-arquitectura/data/2026")
RUTA_GOLD = HISTORICO / "gold_ventanas"
RUTA_MODELO = (SESION / "artifacts") / INTEGRANTE / "modelo_historico"
RUTA_CONTRATO = RUTA_MODELO.parent / "contrato_modelo_historico.json"
CAPTURAS_MINIMAS = 4
INICIO_VALIDACION = "2026-09-01"
INICIO_PRUEBA = "2026-10-01"
SEMILLA = 42

spark = (
    SparkSession.builder.master("local[2]")
    .appName(f"AeroStreamPeru-S05-{INTEGRANTE}")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.driver.memory", "768m")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

manifiesto = HISTORICO / "fuente.json"
if not manifiesto.is_file() or not RUTA_GOLD.is_dir():
    print("Preparando Gold con todos los Parquet montados.", flush=True)
    preparar_historico(spark)
fuente = json.loads(manifiesto.read_text(encoding="utf-8"))
archivos_actuales = sorted(ruta.relative_to(BRONZE).as_posix() for ruta in BRONZE.rglob("*.parquet"))
archivos_del_gold = set(fuente["archivos"])
if set(archivos_actuales) - archivos_del_gold:
    raise ValueError("Hay Parquet locales que Gold no incluye. Regenera Gold.")
if len(archivos_actuales) < len(archivos_del_gold):
    print("Se usa Gold histórico preparado. El Bronze completo no está montado en este equipo.")
print(f"Parquet Bronze procesados: {fuente['archivos_bronze']}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/24 16:48:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/24 16:48:18 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Parquet Bronze procesados: 71


## 3. Transformaciones distribuidas y plan

La celda muestra una fecha real de Bronze para hacer visibles el esquema, los filtros y el plan de Spark. `filter` y `withColumn` definen el plan. `show` inicia la lectura. El entrenamiento usa todo el histórico disponible en los 71 Parquet, a la cadencia de dos minutos de la captura prevista.

In [3]:
primer_archivo = BRONZE / fuente["archivos"][0]
primera_fecha = fuente["primera_ventana_utc"][:10]
inicio_dia = F.unix_timestamp(F.lit(primera_fecha + " 00:00:00"))
primer_dia = (
    spark.read.schema(ESQUEMA_BRONZE).parquet(str(primer_archivo))
    .filter((F.col("time") >= inicio_dia) & (F.col("time") < inicio_dia + 86400))
    .filter(F.pmod(F.col("time"), F.lit(CADENCIA_SEGUNDOS)) == 0)
)
ejemplo_plan = primer_dia.withColumn("captura_utc", F.timestamp_seconds("time"))
ejemplo_plan.explain(True)
ejemplo_plan.select("captura_utc", "icao24", "onground", "velocity").show(5)

muestra = primer_dia.cache()

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(captura_utc, 'timestamp_seconds('time), None)]
+- Filter (pmod(time#0L, cast(120 as bigint)) = cast(0 as bigint))
   +- Filter ((time#0L >= unix_timestamp(2026-01-01 00:00:00, yyyy-MM-dd HH:mm:ss, Some(UTC), true)) AND (time#0L < (unix_timestamp(2026-01-01 00:00:00, yyyy-MM-dd HH:mm:ss, Some(UTC), true) + cast(86400 as bigint))))
      +- Relation [time#0L,icao24#1,lat#2,lon#3,onground#4,velocity#5,baroaltitude#6,vertrate#7,lastposupdate#8,lastcontact#9] parquet

== Analyzed Logical Plan ==
time: bigint, icao24: string, lat: double, lon: double, onground: boolean, velocity: double, baroaltitude: double, vertrate: double, lastposupdate: double, lastcontact: double, captura_utc: timestamp
Project [time#0L, icao24#1, lat#2, lon#3, onground#4, velocity#5, baroaltitude#6, vertrate#7, lastposupdate#8, lastcontact#9, timestamp_seconds(time#0L) AS captura_utc#11]
+- Filter (pmod(time#0L, cast(120 as bigint)) = cast(0 as bigint))
   

+-------------------+------+--------+------------------+
|        captura_utc|icao24|onground|          velocity|
+-------------------+------+--------+------------------+
|2026-01-01 08:52:00|ab2ad8|   false|248.79364976918797|
|2026-01-01 08:54:00|ab2ad8|   false|248.79364976918797|
|2026-01-01 08:56:00|ab2ad8|   false|247.17361263576757|
|2026-01-01 08:58:00|ab2ad8|   false|247.85151667310998|
|2026-01-01 02:26:00|0c20e0|   false|241.69890927549528|
+-------------------+------+--------+------------------+
only showing top 5 rows


## 4. Calidad de Silver y verificación de Gold

Mostramos el antes y después de nulos y duplicados sobre una fecha. La función `limpiar_estados` es la misma que ejecuta la preparación histórica para cada mes. Conserva la velocidad faltante como nulo y resuelve duplicados exactos y conflictos por `(icao24, time)`. En Gold se comprueba el conteo contra el manifiesto y el filtro de partición por fecha.

In [4]:
auditoria = muestra.agg(
    F.count("*").alias("filas"),
    (F.count("*") - F.count("icao24")).alias("sin_icao24"),
    (F.count("*") - F.count("velocity")).alias("sin_velocidad"),
    (F.count("*") - F.count("baroaltitude")).alias("sin_altitud"),
)
auditoria.show()

candidatos, sin_exactos, silver = limpiar_estados(muestra)
silver = silver.cache()
filas_bronze = muestra.count()
filas_validas = candidatos.count()
filas_sin_exactos = sin_exactos.count()
filas_silver = silver.count()
assert filas_silver == silver.select("icao24", "time").distinct().count()
print(f"Filas descartadas por controles obligatorios: {filas_bronze - filas_validas:,}")
print(f"Duplicados exactos: {filas_validas - filas_sin_exactos:,}")
print(f"Conflictos adicionales por (icao24, time): {filas_sin_exactos - filas_silver:,}")

# Contar estados en vuelo con y sin lectura de velocidad con el RDD.
resumen_rdd = (
    silver.filter(~F.col("onground")).select("velocity").rdd
    .map(lambda fila: ("sin lectura" if fila.velocity is None else "con lectura", 1))
    .reduceByKey(lambda a, b: a + b)
    .collect()
)
print(f"Conteo de estados por RDD: {sorted(resumen_rdd)}")
silver.unpersist()
muestra.unpersist()

+-----+----------+-------------+-----------+
|filas|sin_icao24|sin_velocidad|sin_altitud|
+-----+----------+-------------+-----------+
| 4743|         0|          700|        719|
+-----+----------+-------------+-----------+



Filas descartadas por controles obligatorios: 0
Duplicados exactos: 0
Conflictos adicionales por (icao24, time): 0


[Stage 39:>                                                         (0 + 2) / 4]

Conteo de estados por RDD: [('con lectura', 4023), ('sin lectura', 1)]


DataFrame[time: bigint, icao24: string, lat: double, lon: double, onground: boolean, velocity: double, baroaltitude: double, vertrate: double, lastposupdate: double, lastcontact: double]

In [5]:
esquema = (
    StructType()
    .add("inicio_utc", TimestampType())
    .add("fecha_utc", DateType())
    .add("capturas_observadas", LongType())
)
for columna in VARIABLES_PREDICTORAS:
    if columna in {"hora_del_dia", "dia_semana"}:
        tipo = IntegerType()
    elif columna in {"volumen_min_t", "volumen_max_t"}:
        tipo = LongType()
    else:
        tipo = DoubleType()
    esquema.add(columna, tipo)

tipos_gold = {campo.name: campo.dataType for campo in spark.read.parquet(str(RUTA_GOLD)).schema}
incompatibles = [campo.name for campo in esquema if tipos_gold.get(campo.name) != campo.dataType]
assert not incompatibles, f"Regenera Gold de S3. Columnas incompatibles: {incompatibles}"

gold_origen = spark.read.schema(esquema).parquet(str(RUTA_GOLD))
gold_por_fecha = gold_origen.filter(F.col("fecha_utc") == F.lit(primera_fecha).cast("date"))
gold_por_fecha.explain(True)
print(f"Ventanas Gold en {primera_fecha}: {gold_por_fecha.count():,}")
gold = gold_origen.cache()
assert not gold.groupBy("inicio_utc").count().where("count > 1").take(1)
ventanas_gold = gold.count()
assert ventanas_gold == fuente["ventanas"]
print(f"Ventanas de Gold leídas: {ventanas_gold:,}")
gold.groupBy(F.date_format("fecha_utc", "yyyy-MM").alias("mes")).count().orderBy("mes").show()

== Parsed Logical Plan ==
'Filter '`=`('fecha_utc, cast(2026-01-01 as date))
+- Relation [inicio_utc#2215,capturas_observadas#2216L,velocidad_mediana_t#2217,velocidad_p25_t#2218,velocidad_p75_t#2219,altitud_mediana_t#2220,velocidad_vertical_mediana_t#2221,volumen_promedio_t#2222,hora_del_dia#2223,dia_semana#2224,mes#2225,fecha_utc#2226] parquet

== Analyzed Logical Plan ==
inicio_utc: timestamp, capturas_observadas: bigint, velocidad_mediana_t: double, velocidad_p25_t: double, velocidad_p75_t: double, altitud_mediana_t: double, velocidad_vertical_mediana_t: double, volumen_promedio_t: double, hora_del_dia: int, dia_semana: int, mes: string, fecha_utc: date
Filter (fecha_utc#2226 = cast(2026-01-01 as date))
+- Relation [inicio_utc#2215,capturas_observadas#2216L,velocidad_mediana_t#2217,velocidad_p25_t#2218,velocidad_p75_t#2219,altitud_mediana_t#2220,velocidad_vertical_mediana_t#2221,volumen_promedio_t#2222,hora_del_dia#2223,dia_semana#2224,mes#2225,fecha_utc#2226] parquet

== Optimized 

Ventanas de Gold leídas: 46,809
+-------+-----+
|    mes|count|
+-------+-----+
|2026-01| 4242|
|2026-02| 4031|
|2026-03| 4461|
|2026-04| 4319|
|2026-05| 4445|
|2026-06| 4293|
|2026-07| 4461|
|2026-08| 4464|
|2026-09| 4320|
|2026-10| 4461|
|2026-11| 3312|
+-------+-----+



## 5. Construcción de la etiqueta

`lead()` toma el valor posterior en orden temporal. Exigimos que avance exactamente diez minutos para que una ventana ausente no estire el horizonte. Para esta práctica se requieren al menos cuatro de las cinco capturas programadas en ambas ventanas y valores presentes del indicador actual y futuro. Esta cobertura es un criterio de la práctica.

In [6]:
orden_temporal = Window.orderBy("inicio_utc")
con_ventana_siguiente = (
    gold.withColumn("inicio_objetivo_utc", F.lead("inicio_utc").over(orden_temporal))
    .withColumn(VARIABLE_OBJETIVO, F.lead(INDICADOR_ACTUAL).over(orden_temporal))
    .withColumn("capturas_objetivo", F.lead("capturas_observadas").over(orden_temporal))
)
es_contigua = F.col("inicio_objetivo_utc") == F.col("inicio_utc") + F.expr("INTERVAL 10 MINUTES")
pares_contiguos = con_ventana_siguiente.filter(es_contigua).cache()
cobertura_suficiente = (
    (F.col("capturas_observadas") >= CAPTURAS_MINIMAS)
    & (F.col("capturas_objetivo") >= CAPTURAS_MINIMAS)
)
datos_modelo = (
    pares_contiguos.filter(cobertura_suficiente)
    .na.drop(subset=[INDICADOR_ACTUAL, VARIABLE_OBJETIVO])
    .withColumn("fecha_objetivo_utc", F.to_date("inicio_objetivo_utc"))
    .cache()
)
print(f"Ventanas con sucesora contigua: {pares_contiguos.count():,}")
print(f"Pares elegibles para modelar: {datos_modelo.count():,}")
datos_modelo.orderBy("inicio_utc").select(
    "inicio_utc", INDICADOR_ACTUAL, "inicio_objetivo_utc", VARIABLE_OBJETIVO
).show(5, truncate=False)

Ventanas con sucesora contigua: 46,765
Pares elegibles para modelar: 46,614
+-------------------+-------------------+-------------------+--------------------+
|inicio_utc         |velocidad_mediana_t|inicio_objetivo_utc|velocidad_mediana_t1|
+-------------------+-------------------+-------------------+--------------------+
|2026-01-01 00:00:00|238.31253910948703 |2026-01-01 00:10:00|235.1841949016485   |
|2026-01-01 00:10:00|235.1841949016485  |2026-01-01 00:20:00|239.18271451874256  |
|2026-01-01 00:20:00|239.18271451874256 |2026-01-01 00:30:00|236.46747364990946  |
|2026-01-01 00:30:00|236.46747364990946 |2026-01-01 00:40:00|231.6260901346803   |
|2026-01-01 00:40:00|231.6260901346803  |2026-01-01 00:50:00|235.24664072706904  |
+-------------------+-------------------+-------------------+--------------------+
only showing top 5 rows


## 6. División cronológica, imputación y vector

Entrenamos de enero a agosto, comparamos los modelos en septiembre y evaluamos el ganador con octubre y noviembre. Cada ejemplo tiene su ventana de origen y la ventana objetivo en el mismo periodo. Separar por fechas reproduce el uso de un modelo que predice después de entrenar.

El imputador aprende medianas de las predictoras en entrenamiento y `VectorAssembler` produce `features`. La regresión lineal estandariza internamente durante el ajuste. El bosque trabaja con las predictoras en sus escalas originales.

In [7]:
inicio_validacion = F.to_date(F.lit(INICIO_VALIDACION))
inicio_prueba = F.to_date(F.lit(INICIO_PRUEBA))

df_train = datos_modelo.filter(F.col("fecha_objetivo_utc") < inicio_validacion).cache()
df_val = datos_modelo.filter(
    (F.col("fecha_utc") >= inicio_validacion) & (F.col("fecha_objetivo_utc") < inicio_prueba)
).cache()
df_test = datos_modelo.filter(F.col("fecha_utc") >= inicio_prueba).cache()

periodos = {}
for nombre, datos in [("entrenamiento", df_train), ("validación", df_val), ("prueba", df_test)]:
    resumen = datos.agg(
        F.count("*").alias("filas"),
        F.min("inicio_utc").alias("desde"),
        F.max("inicio_objetivo_utc").alias("hasta"),
    ).first()
    assert resumen.filas > 1, f"Periodo sin pares suficientes: {nombre}"
    periodos[nombre] = {
        "pares": resumen.filas,
        "desde_utc": resumen.desde.isoformat(),
        "hasta_utc": resumen.hasta.isoformat(),
    }
    print(f"{nombre}: {resumen.filas:,} pares · {resumen.desde} a {resumen.hasta} UTC")

columnas_imputadas = [f"{columna}_imputada" for columna in VARIABLES_PREDICTORAS]
imputador = Imputer(
    strategy="median", inputCols=VARIABLES_PREDICTORAS, outputCols=columnas_imputadas
)
ensamblador = VectorAssembler(inputCols=columnas_imputadas, outputCol="features")

entrenamiento: 34,531 pares · 2026-01-01 00:00:00 a 2026-08-31 23:50:00 UTC
validación: 4,319 pares · 2026-09-01 00:00:00 a 2026-09-30 23:50:00 UTC
prueba: 7,762 pares · 2026-10-01 00:00:00 a 2026-11-23 23:50:00 UTC


## 7. Modelos candidatos y comparación

Ajustamos una regresión lineal base, tres configuraciones regularizadas y un bosque aleatorio. Calculamos primero la persistencia, que repite el valor de la ventana actual. Comparamos los cinco modelos y esa referencia sobre la misma validación con RMSE, MAE y R². RMSE penaliza más los errores grandes y decide el ganador. MAE sirve como desempate. La importancia del bosque indica qué variables usó para reducir el error al dividir los datos.

In [8]:
def medir(predicciones):
    resultados = {}
    for metrica in ["rmse", "mae", "r2"]:
        evaluador = RegressionEvaluator(
            labelCol=VARIABLE_OBJETIVO, predictionCol="prediction", metricName=metrica
        )
        resultados[metrica] = evaluador.evaluate(predicciones)
    return resultados


persistencia_validacion = df_val.select(
    VARIABLE_OBJETIVO, F.col(INDICADOR_ACTUAL).alias("prediction")
)
referencia_validacion = {
    "modelo": "Persistencia", "regParam": None, "elasticNetParam": None, **medir(persistencia_validacion)
}
print(f"RMSE de persistencia en validación: {referencia_validacion['rmse']:.4f} {UNIDAD}", flush=True)

configuraciones_lineales = [
    ("Lineal base", 0.0, 0.0),
    ("Ridge", 0.1, 0.0),
    ("Lasso", 0.1, 1.0),
    ("Elastic Net", 0.1, 0.5),
]
candidatos = []
for nombre, reg_param, elastic_net in configuraciones_lineales:
    estimador = LinearRegression(
        labelCol=VARIABLE_OBJETIVO, featuresCol="features",
        regParam=reg_param, elasticNetParam=elastic_net,
        maxIter=80, standardization=True,
    )
    candidatos.append((nombre, estimador, reg_param, elastic_net))

bosque = RandomForestRegressor(
    labelCol=VARIABLE_OBJETIVO, featuresCol="features",
    numTrees=24, maxDepth=4, seed=SEMILLA,
)
candidatos.append(("Bosque aleatorio", bosque, None, None))

preprocesador = Pipeline(stages=[imputador, ensamblador]).fit(df_train)
train_con_vector = preprocesador.transform(df_train).select("features", VARIABLE_OBJETIVO).cache()
validacion_con_vector = preprocesador.transform(df_val).select("features", VARIABLE_OBJETIVO).cache()

modelos = {}
resultados = []
for nombre, estimador, reg_param, elastic_net in candidatos:
    modelo = estimador.fit(train_con_vector)
    modelos[nombre] = modelo
    predicciones = modelo.transform(validacion_con_vector).select(VARIABLE_OBJETIVO, "prediction").cache()
    resultados.append({
        "modelo": nombre,
        "regParam": reg_param,
        "elasticNetParam": elastic_net,
        **medir(predicciones),
    })
    predicciones.unpersist()
    print(f"Evaluado: {nombre}", flush=True)

train_con_vector.unpersist()
validacion_con_vector.unpersist()

spark.createDataFrame([*resultados, referencia_validacion]).select(
    "modelo", "regParam", "elasticNetParam", "rmse", "mae", "r2"
).orderBy("rmse", "mae", "modelo").show(truncate=False)

mejor = min(resultados, key=lambda fila: (fila["rmse"], fila["mae"], fila["modelo"]))
nombre_ganador = mejor["modelo"]
modelo_ganador = PipelineModel(stages=[*preprocesador.stages, modelos[nombre_ganador]])
print(f"Ganador de MLlib en validación: {nombre_ganador}")
preferido = nombre_ganador if mejor["rmse"] < referencia_validacion["rmse"] else "Persistencia"
print(f"Preferido según el RMSE de validación: {preferido}")

pesos = modelos["Bosque aleatorio"].featureImportances.toArray()
importancia = [(columna, float(peso)) for columna, peso in zip(VARIABLES_PREDICTORAS, pesos)]
spark.createDataFrame(importancia, ["predictora", "importancia"]).orderBy(
    F.col("importancia").desc(), "predictora"
).show(truncate=False)

RMSE de persistencia en validación: 8.7107 m/s


netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory


Evaluado: Lineal base
Evaluado: Ridge
Evaluado: Lasso
Evaluado: Elastic Net
Evaluado: Bosque aleatorio
+----------------+--------+---------------+-----------------+-----------------+------------------+
|modelo          |regParam|elasticNetParam|rmse             |mae              |r2                |
+----------------+--------+---------------+-----------------+-----------------+------------------+
|Lasso           |0.1     |1.0            |8.047815684372257|5.306279675915811|0.5849979602403483|
|Elastic Net     |0.1     |0.5            |8.056051711910653|5.329958031775268|0.5841481104640407|
|Ridge           |0.1     |0.0            |8.06436913851907 |5.359294779281987|0.5832889791640941|
|Lineal base     |0.0     |0.0            |8.071332011580548|5.359910231682159|0.582569081925002 |
|Bosque aleatorio|NULL    |NULL           |8.572613276730053|5.668076204641968|0.5291087142000122|
|Persistencia    |NULL    |NULL           |8.710739238612856|5.476906738931925|0.5138120232660607|
+-----

## 8. Evaluación de negocio en prueba

Medimos al ganador una sola vez en la prueba reservada. La persistencia usa el valor de la ventana actual como pronóstico. Esta referencia ayuda a interpretar si el modelo mejora una regla sencilla. El MAE muestra cuánto se desvía habitualmente la predicción en m/s. La prueba cubre los dos últimos meses disponibles. Si el modelo pierde frente a persistencia, se conserva ese resultado para la sustentación y se reporta cuál sería la opción operativa según la validación.

In [9]:
predicciones_prueba = modelo_ganador.transform(df_test).select(
    "inicio_utc", "inicio_objetivo_utc", VARIABLE_OBJETIVO, "prediction"
).cache()
referencia_prueba = df_test.select(VARIABLE_OBJETIVO, F.col(INDICADOR_ACTUAL).alias("prediction")).cache()

metricas_ganador = medir(predicciones_prueba)
metricas_referencia = medir(referencia_prueba)
resumen_prueba = [
    (nombre_ganador, metricas_ganador["rmse"], metricas_ganador["mae"], metricas_ganador["r2"]),
    ("Persistencia", metricas_referencia["rmse"], metricas_referencia["mae"], metricas_referencia["r2"]),
]
spark.createDataFrame(resumen_prueba, ["modelo", "rmse", "mae", "r2"]).show(truncate=False)

predicciones_prueba.orderBy("inicio_objetivo_utc").withColumn(
    "error_absoluto", F.abs(F.col("prediction") - F.col(VARIABLE_OBJETIVO))
).show(8, truncate=False)
diferencia_rmse = metricas_referencia["rmse"] - metricas_ganador["rmse"]
print(f"MAE del ganador en prueba: {metricas_ganador['mae']:.3f} {UNIDAD}")
if diferencia_rmse > 0:
    print(f"El ganador reduce el RMSE de persistencia en {diferencia_rmse:.3f} {UNIDAD}.")
else:
    print("El ganador no reduce el RMSE de persistencia en la prueba reservada.")

+------------+------------------+-----------------+-------------------+
|modelo      |rmse              |mae              |r2                 |
+------------+------------------+-----------------+-------------------+
|Lasso       |11.168624062964998|6.752626124361706|0.540704370662759  |
|Persistencia|12.033470426599989|6.896674870185894|0.46681888677616856|
+------------+------------------+-----------------+-------------------+

+-------------------+-------------------+--------------------+------------------+------------------+
|inicio_utc         |inicio_objetivo_utc|velocidad_mediana_t1|prediction        |error_absoluto    |
+-------------------+-------------------+--------------------+------------------+------------------+
|2026-10-01 00:00:00|2026-10-01 00:10:00|236.7230710046779   |230.57843935013074|6.144631654547169 |
|2026-10-01 00:10:00|2026-10-01 00:20:00|232.59753590940468  |233.40632955654115|0.8087936471364685|
|2026-10-01 00:20:00|2026-10-01 00:30:00|224.05320796581947  |

## 9. Guardado y verificación

Se guarda el ganador de validación como `PipelineModel`, con la imputación, el vector de entrada y el estimador ajustados. Después se recarga y se compara la predicción de cada fila de prueba. También verificamos que el modelo pueda puntuar una fila JSON que solo trae las predictoras de una ventana. El contrato JSON enumera las columnas y tipos que tendrá que entregar Spark antes de enviar el resultado al tablero.

In [10]:
RUTA_MODELO.parent.mkdir(parents=True, exist_ok=True)
modelo_ganador.write().overwrite().save(str(RUTA_MODELO))
recargado = PipelineModel.load(str(RUTA_MODELO))

predicciones_recargadas = recargado.transform(df_test).select(
    "inicio_utc", F.col("prediction").alias("prediccion_recargada")
)
comparacion = predicciones_prueba.join(predicciones_recargadas, "inicio_utc")
control = comparacion.agg(
    F.count("*").alias("filas"),
    F.max(F.abs(F.col("prediction") - F.col("prediccion_recargada"))).alias("diferencia_maxima"),
).first()
assert control.filas == periodos["prueba"]["pares"]
assert control.diferencia_maxima < 1e-10
print(f"Modelo guardado: {nombre_ganador}")
print(f"Ruta: {RUTA_MODELO}")
print(f"Predicciones verificadas al recargar: {control.filas}")

# Una ventana agregada como la que recibirá el consumidor de inferencia.
entrada = df_test.orderBy("inicio_utc").select("inicio_utc", *VARIABLES_PREDICTORAS).first()
valores = {columna: entrada[columna] for columna in VARIABLES_PREDICTORAS}
esquema_evento = StructType(
    [df_test.schema[columna] for columna in VARIABLES_PREDICTORAS]
)
evento_json = (
    spark.range(1)
    .select(F.from_json(F.lit(json.dumps(valores, allow_nan=False)), esquema_evento).alias("ventana"))
    .select("ventana.*")
)
prediccion_evento = recargado.transform(evento_json).select("prediction").first().prediction
prediccion_offline = predicciones_prueba.filter(
    F.col("inicio_utc") == F.lit(entrada.inicio_utc)
).select("prediction").first().prediction
assert abs(prediccion_evento - prediccion_offline) < 1e-10
print(f"Inferencia con una ventana JSON: {prediccion_evento:.4f} {UNIDAD}")

contrato = {
    "version": 1,
    "integrante": INTEGRANTE,
    "indicador_actual": INDICADOR_ACTUAL,
    "variable_objetivo": VARIABLE_OBJETIVO,
    "unidad": UNIDAD,
    "ruta_modelo": str(RUTA_MODELO),
    "estimador_mllib": nombre_ganador,
    "preferido_en_validacion": preferido,
    "variables_predictoras": [
        {"nombre": campo.name, "tipo_spark": campo.dataType.simpleString()}
        for campo in esquema_evento
    ],
    "inicio_ventana": "inicio_utc",
    "campo_prediccion": "prediction",
    "horizonte_minutos": fuente["ventana_minutos"],
    "cadencia_segundos": fuente["cadencia_segundos"],
    "alineacion_historica": "time Unix múltiplo de la cadencia en UTC",
    "zona_horaria_variables_calendario": "America/Lima",
    "capturas_minimas": CAPTURAS_MINIMAS,
    "rectangulo": fuente["rectangulo"],
    "rango_velocidad_m_s": fuente["rango_velocidad_m_s"],
    "archivos_bronze": fuente["archivos_bronze"],
    "ventanas_gold": ventanas_gold,
    "periodos": periodos,
    "validacion": [*resultados, referencia_validacion],
    "prueba": {"modelo": metricas_ganador, "persistencia": metricas_referencia},
}
RUTA_CONTRATO.write_text(json.dumps(contrato, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(f"Contrato verificado: {RUTA_CONTRATO}")

Modelo guardado: Lasso
Ruta: /opt/s05-evaluacion-unidad-1/artifacts/joel/modelo_historico
Predicciones verificadas al recargar: 7762
Inferencia con una ventana JSON: 230.5784 m/s
Contrato verificado: /opt/s05-evaluacion-unidad-1/artifacts/joel/contrato_modelo_historico.json


## 10. Reflexión para la defensa

Comparar candidatos evita decidir por lo bien que un modelo se ajusta al pasado. El RMSE hace visibles los errores grandes y el MAE expresa el tamaño medio del error en la unidad del indicador. R² aporta una comparación frente a una predicción constante basada en el promedio. La prueba reservada permite ver cuánto cambia el resultado al pasar a un periodo posterior. Si un candidato más complejo aporta poca mejora, también importa el costo de mantenerlo.

**Hallazgo del histórico.** Lasso obtuvo el menor RMSE entre los estimadores de MLlib en septiembre (8.048 m/s). La persistencia obtuvo 8.711. Según la validación, la opción preferida es Lasso. En octubre–noviembre, el RMSE del modelo fue 11.169 y el de persistencia 12.033.

Para el informe individual, registra la celda de la etiqueta, las fechas de cada conjunto, la tabla de candidatos, la importancia del bosque, el resultado de prueba y la verificación del modelo recargado. La guía pide reloj y perfil visibles en las capturas.

In [11]:
spark.stop()